# Cemiplimab (Libtayo®): Regulatory History, Subgroup Efficacy, and Safety Analysis

This notebook performs reproducible analysis and visualization for Phase 2: Clinical Development and Regulatory Affairs of cemiplimab in oncology, specifically:
1. **Regulatory Approval Timelines**: Mapping FDA and EMA approvals across cSCC, BCC, and NSCLC.
2. **Subgroup Efficacy (EMPOWER-CSCC-1)**: Visualizing Objective Response Rate (ORR) across key patient subgroups (Age, Setting, and ECOG Performance Status).
3. **Safety & Adverse Event Profiles**: Comparing common and severe toxicities in advanced disease and adjuvant cSCC (C-POST Cemiplimab vs Placebo).

## Setup and Imports

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime

# Set style for publication-quality figures
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16,
    'savefig.bbox': 'tight',
    'savefig.dpi': 300
})

# Create figures directory if it doesn't exist
os.makedirs('../figures', exist_ok=True)
print("Setup complete.")

## Part 1: Regulatory Timeline Visualization
We load the regulatory approvals history for cemiplimab in the US (FDA) and EU (EMA).

In [ ]:
reg_df = pd.read_csv('../data/cemiplimab_regulatory_history.csv')
reg_df['approval_date'] = pd.to_datetime(reg_df['approval_date'])
reg_df = reg_df.sort_values(by='approval_date')
reg_df

### Plot 1: Regulatory Approval History Timeline

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

# Define y positions for different indications to separate them clearly
indication_map = {
    'Advanced cSCC': 4,
    'Advanced BCC': 3,
    'Advanced NSCLC': 2,
    'Adjuvant cSCC': 1
}

reg_df['y'] = reg_df['indication'].map(indication_map)

# Plot FDA and EMA points
fda_points = reg_df[reg_df['agency'] == 'FDA']
ema_points = reg_df[reg_df['agency'] == 'EMA']

ax.scatter(fda_points['approval_date'], fda_points['y'], color='#1f77b4', s=250, label='FDA Approved', zorder=3, edgecolors='black')
ax.scatter(ema_points['approval_date'], ema_points['y'], color='#ff7f0e', s=250, label='EMA Approved', zorder=3, edgecolors='black')

# Annotate each point
for idx, row in reg_df.iterrows():
    align = 'left'
    offset = 12
    # Adjust annotation text layout depending on overlaps
    if row['agency'] == 'EMA':
        align = 'right'
        offset = -12
        
    ax.annotate(
        f"{row['agency']} ({row['approval_date'].strftime('%b %Y')})\n{row['clinical_setting']}",
        (row['approval_date'], row['y']),
        textcoords="offset points",
        xytext=(offset, -10),
        ha=align, 
        va='top',
        fontsize=9,
        fontweight='semibold',
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8, ec="gray", lw=0.5)
    )

# Beautify plot
ax.set_ylim(0.5, 4.5)
ax.set_yticks(list(indication_map.values()))
ax.set_yticklabels(list(indication_map.keys()), fontweight='bold', fontsize=12)
ax.set_xlabel('Approval Year', fontweight='bold', fontsize=12)
ax.set_title('Cemiplimab (Libtayo®): FDA & EMA Indication Evolution Timeline', fontweight='bold', pad=20, fontsize=15)
ax.legend(loc='upper right', frameon=True, shadow=True)

# Horizontal guidelines for each indication category
for y in indication_map.values():
    ax.axhline(y, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

plt.xlim(datetime(2018, 1, 1), datetime(2026, 6, 1))
plt.tight_layout()
plt.savefig('../figures/cemiplimab_regulatory_timeline.png', dpi=300)
plt.show()

## Part 2: Subgroup Efficacy (EMPOWER-CSCC-1)
We load subgroup response rates (ORR, CR, PR) from the pivotal advanced cSCC study.

In [ ]:
subgroups_df = pd.read_csv('../data/empower_cscc_1_subgroups.csv')
subgroups_df

### Plot 2: EMPOWER-CSCC-1 Subgroup Efficacy (ORR, CR, PR)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

y_labels = subgroups_df['subgroup_type'] + ': ' + subgroups_df['category']
y_pos = np.arange(len(y_labels))

cr_rates = subgroups_df['cr_pct']
pr_rates = subgroups_df['pr_pct']

bar_width = 0.55

# Stacked horizontal bars
bars_cr = ax.barh(y_pos, cr_rates, color='#2ca02c', edgecolor='black', height=bar_width, label='Complete Response (CR)')
bars_pr = ax.barh(y_pos, pr_rates, left=cr_rates, color='#bcbd22', edgecolor='black', height=bar_width, label='Partial Response (PR)')

ax.set_yticks(y_pos)
ax.set_yticklabels(y_labels, fontweight='bold')
ax.invert_yaxis()  # top-down view
ax.set_xlabel('Objective Response Rate (%)', fontweight='bold')
ax.set_title('EMPOWER-CSCC-1: Efficacy (ORR) Subgroup Analysis\n(Independent Central Review)', fontweight='bold', pad=15)
ax.set_xlim(0, 75)

# Add value labels
for i in range(len(y_labels)):
    orr = subgroups_df.loc[i, 'orr_pct']
    n = subgroups_df.loc[i, 'n_evaluable']
    cr = subgroups_df.loc[i, 'cr_pct']
    pr = subgroups_df.loc[i, 'pr_pct']
    
    # Label total ORR
    ax.text(orr + 1.0, i, f"{orr}% (n={n})", va='center', ha='left', fontweight='bold', color='black')
    # Label CR and PR segments internally if size permits
    if cr > 5:
        ax.text(cr/2, i, f"{cr}%", va='center', ha='center', color='white', fontweight='bold', fontsize=9)
    if pr > 5:
        ax.text(cr + pr/2, i, f"{pr}%", va='center', ha='center', color='black', fontweight='bold', fontsize=9)

ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../figures/empower_subgroups_efficacy.png', dpi=300)
plt.show()

## Part 3: Safety & Adverse Events Profile
We compare safety profiles, focusing on the adjuvant setting (C-POST Cemiplimab vs Placebo) to examine toxicities in a randomized context.

In [ ]:
safety_df = pd.read_csv('../data/cemiplimab_safety_ae_profile.csv')
safety_df

### Plot 3: Adjuvant Safety Comparison (C-POST Cemiplimab vs Placebo)

In [ ]:
# Filter dataset for Adjuvant C-POST
cpost_safety = safety_df[safety_df['trial_context'].str.contains('Adjuvant')]

# Pivot dataset for plotting
cemi_safety = cpost_safety[cpost_safety['trial_context'] == 'Adjuvant cSCC (C-POST Cemiplimab)']
plac_safety = cpost_safety[cpost_safety['trial_context'] == 'Adjuvant cSCC (C-POST Placebo)']

cemi_safety = cemi_safety.set_index('adverse_event')
plac_safety = plac_safety.set_index('adverse_event')

common_aes = list(cemi_safety.index)
x = np.arange(len(common_aes))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

# Plot Bars for All Grades
rects1 = ax.bar(x - width/2, cemi_safety['all_grades_pct'], width, label='Adjuvant Cemiplimab (All Grades)', color='#1f77b4', edgecolor='black')
rects2 = ax.bar(x + width/2, plac_safety['all_grades_pct'], width, label='Placebo (All Grades)', color='#7f7f7f', edgecolor='black')

# Plot overlays/markers for Grade 3-4 (severe)
rects1_sev = ax.bar(x - width/2, cemi_safety['grade_3_4_pct'], width, color='#d62728', edgecolor='black', alpha=0.85, label='Adjuvant Cemiplimab (Grade 3-4)')
rects2_sev = ax.bar(x + width/2, plac_safety['grade_3_4_pct'], width, color='#b22222', edgecolor='black', alpha=0.5, label='Placebo (Grade 3-4)')

ax.set_ylabel('Percentage of Patients (%)', fontweight='bold')
ax.set_title('C-POST Adjuvant Trial: Safety Profile & Toxicity Rates\n(Cemiplimab vs Placebo)', fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(common_aes, fontweight='bold', rotation=15)
ax.set_ylim(0, 40)
ax.legend(loc='upper right', frameon=True, shadow=True)

# Annotate bars
def label_bars(rects, is_cemi=True):
    for rect in rects:
        height = rect.get_height()
        if height > 0:
            ax.annotate(f'{height}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),  # 3 points vertical offset
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=8, fontweight='bold')

label_bars(rects1)
label_bars(rects2)

plt.tight_layout()
plt.savefig('../figures/cemiplimab_safety_comparison.png', dpi=300)
plt.show()